In [1]:
## config
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

con = duckdb.connect("healthcare.duckdb")

con.execute("PRAGMA threads=8")
con.execute("SET memory_limit='6GB'")

import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"

fm.fontManager.addfont(font_path)

font_name = fm.FontProperties(fname=font_path).get_name()

plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print(font_name)

Noto Sans CJK JP


In [2]:
con.execute("show tables;").df()

,name
0,analysis_years
1,atc234_sick
2,atc3_sick
3,atc4_disease_breadth
4,atc4_disease_contribution
5,atc4_disease_growth
6,atc4_disease_growth_rank
7,atc4_disease_hhi
8,atc4_disease_prior
9,atc4_disease_top3


In [8]:
con.execute("""
CREATE OR REPLACE VIEW analysis_years AS SELECT MAX(year) AS latest_year, MAX(year) - 1 AS prev_year, MAX(year) - 2 AS base_year FROM ( SELECT DISTINCT CAST(LEFT(CAST(diagYm AS VARCHAR), 4) AS INTEGER) AS year FROM atc4_sick );
CREATE OR REPLACE VIEW atc4_market_yearly AS 
SELECT CAST(LEFT(CAST(diagYm AS VARCHAR), 4) AS INTEGER) AS year, 
atcStep4Cd, atcStep4CdNm, 
SUM(msupUseAmt) AS use_amt, 
SUM(totUseQty) AS use_qty FROM atc4_sick 
GROUP BY 1, 2, 3;
""")

con.execute("""
-- size
CREATE OR REPLACE VIEW atc4_market_growth AS 
WITH y AS ( 
SELECT m.atcStep4Cd, m.atcStep4CdNm, 
SUM( CASE WHEN m.year = a.base_year THEN m.use_amt ELSE 0 END ) AS base_amt, 
SUM( CASE WHEN m.year = a.prev_year THEN m.use_amt ELSE 0 END ) AS prev_amt, 
SUM( CASE WHEN m.year = a.latest_year THEN m.use_amt ELSE 0 END ) AS latest_amt 
FROM atc4_market_yearly m 
CROSS JOIN analysis_years a 
GROUP BY 1,2
) 
SELECT *, latest_amt - base_amt AS growth_amt, 
(latest_amt - base_amt) / NULLIF(base_amt, 0) AS cumulative_growth_rate, 
POWER( latest_amt / NULLIF(base_amt, 0), 1.0 / 2 ) - 1 AS cagr 
FROM y WHERE base_amt > 0;

CREATE OR REPLACE VIEW atc4_market_yoy AS
WITH yearly AS (
    SELECT
        CAST(LEFT(CAST(diagYm AS VARCHAR), 4) AS INTEGER) AS year,
        atcStep4Cd,
        atcStep4CdNm,
        SUM(msupUseAmt) AS use_amt
    FROM atc4_sick
    GROUP BY
        1, 2, 3
)

SELECT
    *,
    use_amt /
        NULLIF(
            LAG(use_amt) OVER (
                PARTITION BY atcStep4Cd
                ORDER BY year
            ),
            0
        ) - 1 AS yoy_growth
FROM yearly;

-- 지속성 지표
CREATE OR REPLACE VIEW atc4_persistence AS
SELECT
    atcStep4Cd,
    atcStep4CdNm,

    SUM(
        CASE
            WHEN yoy_growth > 0 THEN 1
            ELSE 0
        END
    ) AS positive_growth_periods,

    COUNT(yoy_growth) AS growth_periods,

    AVG(yoy_growth) AS avg_yoy_growth

FROM atc4_market_yoy
GROUP BY
    1,2;
""")

# 3개년 데이터에서는 growth_periods성장기간이 최대 2개이므로:
# 2/2 = 지속 성장
# 1/2 = 혼합
# 0/2 = 지속 감소

In [9]:
con.execute("""
-- 절대적인 임계값을 임의로 정하지 않고 Quartile을 사용한다.
CREATE OR REPLACE VIEW atc4_market_position AS
SELECT
    g.*,
    p.positive_growth_periods,

    NTILE(4) OVER (
        ORDER BY latest_amt
    ) AS size_quartile,

    NTILE(4) OVER (
        ORDER BY cumulative_growth_rate
    ) AS growth_quartile,

    NTILE(4) OVER (
        ORDER BY positive_growth_periods
    ) AS persistence_quartile
FROM atc4_market_growth g 
LEFT JOIN atc4_persistence p 
ON g.atcStep4Cd = p.atcStep4Cd;
""")
# 임계값을 Quartile로 정하는 이유
# Median: 큰/작은 시장의 2분할
# Quartile: 시장을 4개 포지션으로 구분
# Quintile: 상위 20% 후보 선정에 적합하지만 작은 표본에서 과도한 정밀성을 줄 수 있음

# 따라서 기본 분석 = Quartile,
# 민감도 분석 = Median / Quintile로 한다.

con.execute("""
CREATE OR REPLACE VIEW atc4_market_share AS
WITH yearly AS (
    SELECT
        year,
        atcStep4Cd,
        atcStep4CdNm,
        use_amt,

        SUM(use_amt) OVER (
            PARTITION BY year
        ) AS total_market
    FROM atc4_market_yearly
),

share AS (
    SELECT
        *,
        use_amt / NULLIF(total_market, 0) AS market_share
    FROM yearly
)

SELECT
    atcStep4Cd,
    atcStep4CdNm,

    MAX(
        CASE
            WHEN year = (SELECT base_year FROM analysis_years)
            THEN market_share
        END
    ) AS base_share,

    MAX(
        CASE
            WHEN year = (SELECT latest_year FROM analysis_years)
            THEN market_share
        END
    ) AS latest_share

FROM share
GROUP BY
    1,2;

-- 최종 변화:

CREATE OR REPLACE VIEW atc4_share_change AS
SELECT
    *,
    latest_share - base_share AS share_change
FROM atc4_market_share;
""")

In [13]:
con.execute("""
CREATE OR REPLACE VIEW atc4_disease_yearly aS 
SELECT 
CAST(LEFT(CAST(diagYm AS VARCHAR), 4) AS INTEGER) AS year, 
atcStep4Cd, atcStep4CdNm, st3SickSym, st3SickSymNm, 
SUM(msupUseAmt) AS use_amt FROM atc4_sick 
GROUP BY
1,2,3,4,5
;

CREATE OR REPLACE VIEW atc4_disease_growth AS 
WITH x AS ( 
SELECT 
d.atcStep4Cd, d.atcStep4CdNm, d.st3SickSym, d.st3SickSymNm, 
SUM( CASE WHEN d.year = a.base_year THEN d.use_amt ELSE 0 END ) AS base_amt, 
SUM( CASE WHEN d.year = a.prev_year THEN d.use_amt ELSE 0 END ) AS prev_amt, 
SUM( CASE WHEN d.year = a.latest_year THEN d.use_amt ELSE 0 END ) AS latest_amt 
FROM atc4_disease_yearly d 
cROSS JOIN analysis_years a 
GROUP BY 
1,2,3,4 
) 
SELECT *, latest_amt - base_amt AS growth_amt, 
(latest_amt - base_amt) / NULLIF(base_amt, 0) AS cumulative_growth_rate, 
POWER( latest_amt / NULLIF(base_amt, 0), 1.0 / 2 ) - 1 AS cagr 
FROM x 
WHERE base_amt > 0
;
CREATE OR REPLACE VIEW atc4_disease_contribution AS 
SELECT *, SUM(growth_amt) OVER ( PARTITION BY atcStep4Cd ) AS atc4_growth_amt, 
growth_amt / NULLIF( SUM(growth_amt) OVER ( PARTITION BY atcStep4Cd ), 0 ) AS growth_contribution 
FROM atc4_disease_growth;


-- 상병 성장집중도
CREATE OR REPLACE VIEW atc4_disease_growth_rank AS
SELECT
    *,
    ROW_NUMBER() OVER (
        PARTITION BY atcStep4Cd
        ORDER BY growth_amt DESC
    ) AS growth_rank

FROM atc4_disease_contribution;

-- Top 3 성장기여:

CREATE OR REPLACE VIEW atc4_disease_top3 AS
SELECT
    atcStep4Cd,
    atcStep4CdNm,

    SUM(
        CASE
            WHEN growth_rank <= 3
            THEN growth_contribution
            ELSE 0
        END
    ) AS top3_growth_contribution

FROM atc4_disease_growth_rank

GROUP BY
    1,2
    ;

-- 성장폭
CREATE OR REPLACE VIEW atc4_disease_breadth AS 
SELECT atcStep4Cd, atcStep4CdNm, 
COUNT(*) FILTER ( WHERE growth_amt > 0 ) AS growing_disease_count, 
COUNT(*) FILTER ( WHERE growth_amt < 0 ) AS declining_disease_count, 
COUNT(*) AS total_disease_count, COUNT(*) FILTER ( WHERE growth_amt > 0 ) * 1.0 / NULLIF(COUNT(*), 0) AS growth_breadth 
FROM atc4_disease_growth 
GROUP BY 1,2;


""")

con.execute("""
-- atc4의 상병사전분포도 // 최근 연도 기준으로 ATC4 내부에서 각 상병이 차지하는 비중이다.

CREATE OR REPLACE VIEW atc4_disease_prior AS
SELECT
    atcStep4Cd,
    atcStep4CdNm,
    st3SickSym,
    st3SickSymNm,

    latest_amt,

    latest_amt /
        NULLIF(
            SUM(latest_amt) OVER (
                PARTITION BY atcStep4Cd
            ),
            0
        ) AS disease_share,

    growth_amt,
    cumulative_growth_rate,
    cagr

FROM atc4_disease_growth
;

CREATE OR REPLACE VIEW atc4_disease_hhi AS 
SELECT 
atcStep4Cd, atcStep4CdNm, SUM( POWER(disease_share, 2) ) AS disease_hhi 
FROM atc4_disease_prior 
GROUP BY 1,2
;

""")



In [15]:
con.execute("""
CREATE OR REPLACE VIEW atc4_market_opportunity AS 
SELECT p.atcStep4Cd, p.atcStep4CdNm, p.base_amt, p.prev_amt, p.latest_amt, 
p.growth_amt, p.cumulative_growth_rate, p.cagr, 
p.size_quartile, p.growth_quartile, p.persistence_quartile, 
ps.positive_growth_periods, sc.base_share, sc.latest_share, sc.share_change, 
b.growing_disease_count, b.declining_disease_count, b.growth_breadth,
t.top3_growth_contribution, h.disease_hhi 
FROM atc4_market_position p 
LEFT JOIN atc4_persistence ps ON p.atcStep4Cd = ps.atcStep4Cd 
LEFT JOIN atc4_share_change sc ON p.atcStep4Cd = sc.atcStep4Cd 
LEFT JOIN atc4_disease_breadth b ON p.atcStep4Cd = b.atcStep4Cd 
LEFT JOIN atc4_disease_top3 t ON p.atcStep4Cd = t.atcStep4Cd 
LEFT JOIN atc4_disease_hhi h ON p.atcStep4Cd = h.atcStep4Cd
;
CREATE OR REPLACE VIEW atc4_market_segment AS 
SELECT *, 
CASE 
WHEN size_quartile = 4 AND growth_quartile = 4 AND persistence_quartile = 4 
THEN 'Core Growth Market' 
WHEN growth_quartile = 4 AND size_quartile IN (2,3) 
THEN 'Emerging Growth Market' 
WHEN growth_quartile = 1 AND persistence_quartile = 1 
THEN 'Declining Market' ELSE 'Other Market' END AS market_segment 
FROM atc4_market_opportunity
;
""")